# 5.2 Blind Methods

With the basic understanding for the data distributions over time, we can start looking for concept drifts. 

**Blind methods** detect or adapt to drift without relying on immediate performance feedback—that is, they do **not require ground truth labels** during deployment.
In real-world scenarios, data often arrives continuously, but the true outcomes (labels) may not be available immediately. Let's see how this works out for the disaster data.

In [ ]:
from pathlib import Path

candidate_paths = [
  Path("configs/visualisations.mplstyle"),
  Path("05_Exercise_5/configs/visualisations.mplstyle"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
  raise FileNotFoundError("Could not find configs/visualisations.mplstyle. Start Jupyter in the repository root or 05_Exercise_5.")

print(f"Loading: {data_path.resolve()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src.data_handling import _load_NASA_data
plt.style.use(str(data_path.resolve()))
from src.visual_helper import defined_colors

df = _load_NASA_data()

## **TASK: Baseline Model**

Build a baseline DecisionTreeRegressor model to predict ``Total Damages (US$)`` based on ``Total Deaths`` and ``No Injured``' This model will serve as our reference for evaluating drift detection methods.

Complete the following steps by replacing the ``None`` placeholders with the correct values and variables:
1. Split the data into training (first 70% of years) and testing (last 30% of years)
2. Instantiate a DecisionTreeRegressor (use sklearn's DecisionTreeRegressor [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html))
3. Train the model and evaluate its performance using MAE and RMSE metrics
4. Visualize the deviations of the baseline model predictions from the true values and evaluate the performance in your own words.

In [ ]:
# select the variables for the analysis
disaster_type = "Flood"
features = ["Total Deaths", "No Injured"]
target = "Total Damages (US$)"

# prepare the data for modeling
df_model = df[df["Disaster Type"] == disaster_type].copy()
df_model = df_model[["Year"] + features + [target]].dropna()
df_model = df_model[df_model[target] > 0]

# log-transform features and target to reduce spread (log1p is a robust log-transformation, using the natural logarithm of 1 + x)
for col in features + [target]:
    df_model[col + " (log)"] = np.log1p(df_model[col])
df_model = df_model.sort_values("Year")
X = df_model[[f + " (log)" for f in features]]
y = df_model[target + " (log)"]
years = df_model["Year"].values

In [ ]:
# 1. Split the data into training (first 70% of years) and testing (last 30% of years)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

In [ ]:
# 2. Instantiate a DecisionTreeRegressor
from sklearn.tree import DecisionTreeRegressor

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

In [ ]:
# 3. Train the model and evaluate its performance using MAE and RMSE metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

In [ ]:
# 4. Visualize the deviations of the baseline model predictions from the true values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

plt.tight_layout()
plt.show()

4. Evaluation of the performance in your own words

...

## **TASK: Rolling Window Approach for Retraining**

Now we know that our baseline model struggles with a lot of data points. While we still do not know whether the performance loss stems from concept drift, we could guess that something is happening with the target variable such that outdated data is skewing the more recent predictions of our model. 

Implement a "blind" drift detection method that trains on the past k years and evaluates on the next year, rolling forward through time. This simulates real-world scenarios where models are periodically retrained.

Complete the following steps by replacing the ``None`` placeholders with the correct values and variables:
1. Derive the test and train sets based on a rolling window of 15 years
2. Train a DecisionTreeRegressor and calculate performance metrics
3. Visualize performance over time and compare it to the baseline model

In [ ]:
# train on a rolling window of past 15 years
window_size = 15

# Prepare results storage
results_blind = []

# Get unique years in order
unique_years = sorted(df_model["Year"].unique())

In [ ]:
# Rolling window approach
for i in range(window_size, len(unique_years)):
    
    # (1.) Derive the test and train sets based on a rolling window of years
    # ---- YOUR CODE GOES HERE (START) ---

    # --- YOUR CODE GOES HERE (END) ---
    
    # (2.) Train a DecisionTreeRegressor and calculate performance metrics
    # ---- YOUR CODE GOES HERE (START) ---

    # --- YOUR CODE GOES HERE (END) ---
    
    results_blind.append({
        'Test_Year': test_year,
        'Train_Years': f"{train_years[0]}-{train_years[-1]}",
        'Train_Size': len(X_train),
        'Test_Size': len(X_test),
        'R2': r2,
        'MAE': mae,
        'RMSE': rmse
    })

results_blind_df = pd.DataFrame(results_blind)

print(f"Average Performance:")
print(f"  MAE: {results_blind_df['MAE'].mean():.4f} (±{results_blind_df['MAE'].std():.4f}), baseline: {baseline_mae:.4f}")
print(f"  RMSE: {results_blind_df['RMSE'].mean():.4f} (±{results_blind_df['RMSE'].std():.4f}), baseline: {baseline_rmse:.4f}")

In [ ]:
# (3.) Visualize performance over time and compare it to the baseline model
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

# plot formatting
plt.xlabel('Test Year')
plt.ylabel('MAE')
plt.title('Mean Absolute Error for Damage Prediction Over Time')
plt.legend()
plt.ylim(0, max(results_blind_df['MAE'].max(), baseline_mae) * 1.1)

plt.tight_layout()
plt.show()